# MuRIL Training — Multilingual Hate Speech & Misinformation
**Jay Modhiya | Krishna Nandi | SIT Pune — NLPA + MLOps Project**

Runtime: GPU P100 or T4 x2 | Expected time: ~35 min per task

In [ ]:
# ── Step 1: Install dependencies ──
!pip install transformers datasets mlflow scikit-learn seaborn -q

In [ ]:
# ── Step 2: Clone your GitHub repo ──
# Replace with your actual GitHub repo URL
!git clone https://github.com/Jay-Modhiya/multilingual-hate-misinfo.git
%cd multilingual-hate-misinfo

In [ ]:
# ── Step 3: Upload HASOC data files ──
# Upload hasoc_train.tsv and hasoc_test.tsv via Kaggle sidebar
import os
os.makedirs('data/raw', exist_ok=True)

# Copy from Kaggle input
!cp /kaggle/input/hasoc-hindi/hasoc_train.tsv data/raw/hasoc_train.tsv
!cp /kaggle/input/hasoc-hindi/hasoc_test.tsv  data/raw/hasoc_test.tsv
print('Data ready ✓')

In [ ]:
# ── Step 4: Verify GPU ──
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# ── Step 5: Verify all datasets load ──
from src.data.loader import load_config, load_all_datasets
cfg = load_config('configs/config.yaml')
datasets = load_all_datasets(cfg)
for name, splits in datasets.items():
    print(f"{name}: train={len(splits['train'])}  val={len(splits['val'])}  test={len(splits['test'])}")

In [ ]:
# ── Step 6: Train HATE model ──
from src.training.trainer import train
hate_metrics = train(cfg, task='hate')
print('\nHate model done:', hate_metrics)

In [ ]:
# ── Step 7: Train MISINFO model ──
misinfo_metrics = train(cfg, task='misinfo')
print('\nMisinfo model done:', misinfo_metrics)

In [ ]:
# ── Step 8: Save models + MLflow logs ──
# Zip everything for download
!zip -r saved_models.zip models/saved/ outputs/ mlruns/
print('Download saved_models.zip from Kaggle output panel')

In [ ]:
# ── Step 9: View MLflow results ──
import mlflow
mlflow.set_tracking_uri('mlruns')

client = mlflow.tracking.MlflowClient()
for exp_name in ['muril-multilingual-hate-hate', 'muril-multilingual-hate-misinfo']:
    try:
        exp = client.get_experiment_by_name(exp_name)
        runs = client.search_runs(exp.experiment_id, order_by=['metrics.test_f1 DESC'])
        if runs:
            r = runs[0]
            print(f"\n{exp_name}")
            print(f"  Test F1       : {r.data.metrics.get('test_f1', 'N/A'):.2f}%")
            print(f"  Test Accuracy : {r.data.metrics.get('test_accuracy', 'N/A'):.2f}%")
    except:
        pass